<a href="https://colab.research.google.com/github/JaymeManhica/AndroidFlutter/blob/master/C%C3%B3pia_de_Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# REMOVER SILÊNCIO DE UM ÁUDIO GRANDE — GOOGLE COLAB
# VERSÃO POR PARTES (mais rápida, com progresso visível)
# ============================================================
# Divide o áudio em blocos de N minutos, processa cada bloco
# separadamente (muito mais rápido que processar tudo de uma vez)
# e depois junta tudo num único ficheiro final.
# ============================================================

print("A instalar ffmpeg (se necessário)...")
get_ipython().system('apt-get install -y ffmpeg -q')

from google.colab import files
import subprocess
import os
import math

# ---- Upload do ficheiro ----
print("\n" + "="*60)
print("Selecciona o teu ficheiro de áudio:")
print("="*60)
uploaded = files.upload()
nome_original = list(uploaded.keys())[0]

input_filename = "audio_entrada.aac"
os.rename(nome_original, input_filename)
print(f"\nRenomeado para: {input_filename}")
print(f"Tamanho: {os.path.getsize(input_filename) / (1024*1024):.1f} MB")

# ---- Descobrir a duração total do áudio ----
def obter_duracao_segundos(ficheiro):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        ficheiro
    ]
    out = subprocess.run(cmd, capture_output=True, text=True)
    return float(out.stdout.strip())

duracao_total = obter_duracao_segundos(input_filename)
print(f"Duração total: {duracao_total/60:.1f} minutos ({duracao_total/3600:.2f} horas)")

# ---- Configurações ----
DURACAO_BLOCO_MIN = 15          # tamanho de cada parte, em minutos
stop_duration = "2.0"           # pausas >= 2s são removidas (mais conservador)
stop_threshold = "-35dB"        # sensibilidade ao silêncio (menos agressivo que -40dB)
stop_periods = "-1"             # remove TODAS as ocorrências de silêncio, não só extremos

duracao_bloco_seg = DURACAO_BLOCO_MIN * 60
num_blocos = math.ceil(duracao_total / duracao_bloco_seg)
print(f"O áudio será dividido em {num_blocos} partes de ~{DURACAO_BLOCO_MIN} minutos cada.\n")

# Pasta para guardar os blocos processados
os.makedirs("blocos_processados", exist_ok=True)

filtro = (
    f"silenceremove="
    f"stop_periods={stop_periods}:"
    f"stop_duration={stop_duration}:"
    f"stop_threshold={stop_threshold}:"
    f"detection=peak"
)

ficheiros_processados = []
falhou = False

for i in range(num_blocos):
    inicio = i * duracao_bloco_seg
    saida_bloco = f"blocos_processados/parte_{i:03d}.mp3"

    print(f"[{i+1}/{num_blocos}] A processar minutos {inicio/60:.1f}–{(inicio+duracao_bloco_seg)/60:.1f}...")

    cmd = [
        "ffmpeg", "-y", "-nostdin",
        "-ss", str(inicio),
        "-t", str(duracao_bloco_seg),
        "-i", input_filename,
        "-af", filtro,
        "-c:a", "libmp3lame", "-b:a", "128k",
        saida_bloco
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        if result.returncode != 0:
            print(f"  ERRO na parte {i+1}:")
            print(result.stderr[-1000:])
            falhou = True
            break
        else:
            tamanho = os.path.getsize(saida_bloco) / (1024*1024)
            # Validar se o bloco tem áudio utilizável (duração > 0)
            try:
                dur_bloco = obter_duracao_segundos(saida_bloco)
            except Exception:
                dur_bloco = 0

            if tamanho < 0.01 or dur_bloco < 0.1:
                print(f"  Bloco vazio/sem fala detectada (ignorado) — {tamanho:.2f} MB")
            else:
                print(f"  Concluído ({tamanho:.1f} MB, {dur_bloco:.1f}s)")
                ficheiros_processados.append(saida_bloco)
    except subprocess.TimeoutExpired:
        print(f"  ERRO: parte {i+1} excedeu 5 minutos. A saltar esta parte.")
        falhou = True
        break

# ---- Juntar todas as partes processadas ----
if ficheiros_processados and not falhou:
    print("\nA juntar todas as partes num único ficheiro final...")

    lista_path = "lista_blocos.txt"
    with open(lista_path, "w") as f:
        for fp in ficheiros_processados:
            f.write(f"file '{os.path.abspath(fp)}'\n")

    output_filename = "audio_sem_silencio.mp3"
    cmd_concat = [
        "ffmpeg", "-y", "-nostdin",
        "-f", "concat", "-safe", "0",
        "-i", lista_path,
        "-c", "copy",
        output_filename
    ]
    result = subprocess.run(cmd_concat, capture_output=True, text=True, timeout=300)

    if result.returncode != 0:
        print("ERRO ao juntar as partes:")
        print(result.stderr[-1500:])
    else:
        tamanho_original = os.path.getsize(input_filename) / (1024*1024)
        tamanho_final = os.path.getsize(output_filename) / (1024*1024)
        duracao_final = obter_duracao_segundos(output_filename)

        print(f"\nProcessamento concluído com sucesso!")
        print(f"Duração original: {duracao_total/60:.1f} min")
        print(f"Duração final: {duracao_final/60:.1f} min")
        print(f"Tempo removido: {(duracao_total - duracao_final)/60:.1f} min")
        print(f"Tamanho final: {tamanho_final:.1f} MB")

        print("\nA preparar download...")
        files.download(output_filename)
        print("Concluído!")
elif falhou:
    print("\nO processamento foi interrompido devido a um erro numa das partes.")
    print(f"Partes concluídas com sucesso: {len(ficheiros_processados)}/{num_blocos}")
    print("Podes tentar novamente, ou ajustar DURACAO_BLOCO_MIN para um valor menor (ex: 10).")
else:
    print("\nNenhuma parte foi processada com sucesso.")


# ============================================================
# NOTAS:
# ============================================================
# - DURACAO_BLOCO_MIN: tamanho de cada bloco processado. Se continuar
#   lento, reduz para 10 ou 5 minutos — blocos menores processam mais
#   rápido e têm timeout individual, evitando bloqueios longos.
#
# - stop_threshold: ajusta para -30dB se houver muito ruído de fundo,
#   ou -50dB para ser mais agressivo a remover silêncio.
#
# - stop_duration: aumenta para "1.0" ou "1.5" para preservar mais as
#   pausas naturais da fala.
#
# - Cada bloco tem um timeout de 5 minutos. Se uma parte específica
#   estiver a causar problemas, o script avisa qual e pára aí, em vez
#   de ficar preso indefinidamente.
# ============================================================

A instalar ffmpeg (se necessário)...
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.

Selecciona o teu ficheiro de áudio:


Saving Seminário alusivo ao Dia Internacional dos Arquivos - 2026_06_15 07_38 CAT – Recording.aac to Seminário alusivo ao Dia Internacional dos Arquivos - 2026_06_15 07_38 CAT – Recording.aac

Renomeado para: audio_entrada.aac
Tamanho: 396.6 MB
Duração total: 424.5 minutos (7.08 horas)
O áudio será dividido em 29 partes de ~15 minutos cada.

[1/29] A processar minutos 0.0–15.0...
  Concluído (0.0 MB, 1.6s)
[2/29] A processar minutos 15.0–30.0...
  Concluído (0.1 MB, 8.0s)
[3/29] A processar minutos 30.0–45.0...
  Concluído (0.0 MB, 2.0s)
[4/29] A processar minutos 45.0–60.0...
  Concluído (1.1 MB, 74.0s)
[5/29] A processar minutos 60.0–75.0...
  Concluído (11.7 MB, 768.0s)
[6/29] A processar minutos 75.0–90.0...
  Concluído (12.8 MB, 840.0s)
[7/29] A processar minutos 90.0–105.0...
  Concluído (12.9 MB, 846.0s)
[8/29] A processar minutos 105.0–120.0...
  Concluído (12.6 MB, 828.0s)
[9/29] A processar minutos 120.0–135.0...
  Concluído (12.8 MB, 840.0s)
[10/29] A processar minutos 135.0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Concluído!
